# Lab 16-08 · One Transfer Function, Many Structures

Five parts, each built on `structures.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math, subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import structures as st
from make_vectors import tests, COEF, build
sos = signal.ellip(8, 0.1, 60, 4000, fs=48000, output="sos")

## Part 1 · Equivalence

*Predict:* will DF I, DF II and DF2T differ in floating point by more than 1e-10?

In [2]:
rng = np.random.default_rng(8)
x = rng.standard_normal(2000)
b, a = signal.ellip(4, 0.5, 40, 0.3)
ref = signal.lfilter(b, a, x)
res = {"df1": st.df1(b, a, x), "df2": st.df2(b, a, x)[0], "df2t": st.df2t(b, a, x)[0],
       "cascade": st.sos_filter(signal.tf2sos(b, a), x)}
for k, yv in res.items():
    print(f"{k:8s} max |difference| = {np.max(np.abs(yv - ref)):.2e}")
    assert np.max(np.abs(yv - ref)) < 1e-10
h = rng.standard_normal(16)
assert np.allclose(st.fir_direct(h, x[:300]), np.convolve(h, x[:300])[:300])
assert np.allclose(st.fir_transposed(h, x[:300]), np.convolve(h, x[:300])[:300])

df1      max |difference| = 2.66e-15
df2      max |difference| = 2.66e-15
df2t     max |difference| = 3.77e-15
cascade  max |difference| = 8.44e-15


## Part 2 · Internal signals

*Predict:* the ratio of DF II's state amplitude to the input amplitude for the sharpest section at its worst frequency.

In [3]:
sec = sos[3]
w = np.linspace(0, np.pi, 400001)
_, HA = signal.freqz([1], sec[3:], worN=w)
wpk = w[np.argmax(np.abs(HA))]
n = np.arange(20000)
xs = np.cos(wpk * n)
y, wst = st.df2(sec[:3], sec[3:], xs)
amp_w, amp_y = np.max(np.abs(wst[10000:])), np.max(np.abs(y[10000:]))
print(f"peak frequency {wpk * 48000 / (2 * np.pi):.0f} Hz: state amplitude {amp_w:.2f}, section output {amp_y:.2f}")
assert abs(amp_w - 66.1) < 0.1 and abs(amp_y - 6.17) < 0.01
full = st.sos_filter(sos, xs)
print(f"whole cascade output amplitude {np.max(np.abs(full[10000:])):.3f}")
assert np.max(np.abs(full[10000:])) <= 1.0

peak frequency 4066 Hz: state amplitude 66.10, section output 6.17


whole cascade output amplitude 0.839


## Part 3 · Section ordering

*Predict:* the largest intermediate peak for the reversed ordering.

In [4]:
def peaks(S):
    out, H = [], np.ones_like(w, dtype=complex)
    for s_ in S:
        _, Hs = signal.freqz(s_[:3], s_[3:], worN=w)
        H = H * Hs
        out.append(np.abs(H).max())
    return np.array(out)
pf, pr = peaks(sos), peaks(sos[::-1])
print("SciPy order:", np.round(pf, 3), " reversed:", np.round(pr, 3))
assert np.all(pf <= 1.0 + 1e-9) and abs(pr[2] - 53.57) < 0.01

SciPy order: [0.08  0.386 0.73  1.   ]  reversed: [ 6.25  17.219 53.567  1.   ]


## Part 4 · Pole grids

*Predict:* at 12 fractional bits, which form misses 100 Hz by more?

In [5]:
fs, f0, r = 48000, 100, 0.999
th = 2 * np.pi * f0 / fs
q = lambda v, B: np.round(v * 2.0 ** B) / 2.0 ** B
for B in (12, 16):
    p = np.roots([1, q(-2 * r * np.cos(th), B), q(r * r, B)])
    fd = abs(np.angle(p[0])) * fs / (2 * np.pi)
    fc = math.atan2(q(r * np.sin(th), B), q(r * np.cos(th), B)) * fs / (2 * np.pi)
    print(f"{B} bits: direct form {fd:.2f} Hz, coupled form {fc:.3f} Hz")
    if B == 12:
        assert abs(fd - 119.19) < 0.005 and abs(fc - 100.81) < 0.005
k = st.poly_to_lattice([1, -0.9, 0.64, -0.576])
print("reflection coefficients:", np.round(k, 4))
assert np.allclose(st.lattice_to_poly(k), [1, -0.9, 0.64, -0.576]) and np.all(np.abs(k) < 1)

12 bits: direct form 119.19 Hz, coupled form 100.808 Hz
16 bits: direct form 98.73 Hz, coupled form 100.002 Hz
reflection coefficients: [-0.6728  0.182  -0.576 ]


## Part 5 · The hardware

*Predict:* how many of the 960 expected outputs change if the model truncates instead of rounding?

In [6]:
lines, sat = build()
committed = open("vectors.hex").read().split()
assert lines == committed, "vectors.hex is stale: rerun make_vectors.py"
print(f"vectors.hex matches the model: {len(lines)} words, {sat} saturating outputs")
assert sat == 4
def truncating(x, coef):
    b0, b1, b2, a1, a2 = coef
    x1 = x2 = y1 = y2 = 0
    out = []
    for xn in x:
        acc = b0 * xn + b1 * x1 + b2 * x2 - a1 * y1 - a2 * y2
        yn = max(-32768, min(32767, acc >> 14))
        x2, x1, y2, y1 = x1, xn, y1, yn
        out.append(yn)
    return out
changed = sum(sum(u != v for u, v in zip(st.biquad_df1_fixed(x, COEF)[0], truncating(x, COEF))) for x in tests)
print("outputs changed by truncation:", changed, "of", 64 * len(tests))
assert changed == 895
try:
    import os, tempfile
    vvp = os.path.join(tempfile.mkdtemp(), "bq.vvp")
    subprocess.run(["iverilog", "-g2012", "-o", vvp, "biquad_df1_tb.v", "biquad_df1.v"], capture_output=True, text=True, check=True)
    out = subprocess.run(["vvp", vvp], capture_output=True, text=True).stdout
    print(out.strip().splitlines()[-1])
    assert "TEST PASSED" in out
except FileNotFoundError:
    print("Icarus Verilog not installed here; CI runs the testbench (tools/run_hdl.py).")

vectors.hex matches the model: 1920 words, 4 saturating outputs
outputs changed by truncation: 895 of 960
biquad_df1_tb.v:33: $finish called at 9746000 (1ps)
